# CumulativeTriTopic — Strategy Benchmark

Benchmarks all three `CumulativeTriTopic` strategies on **AG News** (120K docs, 4 topics) with a **temporal drift simulation**:

| Batches | Topics visible |
|---------|---------------|
| 1–3     | World + Sports |
| 4–6     | + Business emerges |
| 7–9     | + Sci/Tech emerges |
| 10–12   | All 4 stable |

**Strategies compared:** `global_refit` · `coreset` · `batch_merge` · `full_batch` (ceiling)

## 1 — Install

In [ ]:
# The 'dependency conflict' warnings below are harmless — they come from Kaggle's
# pre-installed RAPIDS (cuML/cuDF) stack, which pins numba to an older version.
# tritopic does NOT use RAPIDS, so those conflicts have no effect here.

!pip install "tritopic[full] @ git+https://github.com/nevil-mathew/topic-extraction-poc.git@batch-flow" --quiet 2>&1 | grep -E "^(Successfully|ERROR:|Collecting|error)" | head -20
!pip install datasets sentence-transformers umap-learn psutil --quiet 2>&1 | grep -E "^(Successfully|ERROR:|error)" | head -10

# Verify key imports work before proceeding
import importlib
_required = ["tritopic", "datasets", "sentence_transformers", "umap", "psutil", "plotly"]
_missing  = [m for m in _required if importlib.util.find_spec(m) is None]
if _missing:
    raise ImportError(f"Install failed — missing: {_missing}. Re-run this cell.")
print("All required packages installed:", _required)

## 2 — Imports & constants

In [ ]:
import copy
import pickle
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
import psutil
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

from tritopic.core.model import TriTopic, TriTopicConfig
from tritopic.cumulative.cumulative import CumulativeConfig, CumulativeTriTopic
from tritopic.cumulative.evaluation import compare_to_full_batch
from tritopic.utils.metrics import (
    compute_ari,
    compute_nmi,
    compute_silhouette,
    compute_coherence,
    compute_diversity,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

# ── tuneable constants ──────────────────────────────────────────────────────
EMBED_MODEL      = "all-MiniLM-L6-v2"   # swap to all-mpnet-base-v2 for richer embeddings
EMBED_BATCH_SIZE = 512
CACHE_DIR        = Path("/kaggle/working/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
EMB_CACHE        = CACHE_DIR / f"agnews_emb_{EMBED_MODEL.replace('/', '_')}.npy"
LABELS_CACHE     = CACHE_DIR / "agnews_labels.npy"
DOCS_CACHE       = CACHE_DIR / "agnews_docs.pkl"

NOVELTY_THRESHOLD = 0.30
# Fraction-based min_cluster_size: keeps ~0.5% of the current fitting corpus
# as the minimum community size. Scales automatically from small early batches
# (10K → mcs=50) to the full accumulator (120K → mcs=600), so you never need
# to tune this number manually. The absolute floor of 5 prevents degenerate
# behaviour on tiny test corpora.
MIN_CLUSTER_FRACTION = 0.005   # 0.5% of corpus — tune between 0.002 and 0.01
MIN_CLUSTER_SIZE_ABS = 5       # absolute floor (used only for very small corpora)

N_CONSENSUS_RUNS = 5
RANDOM_STATE     = 42
N_BATCHES        = 12
BATCH_SIZE       = 10_000

STRATEGIES  = ["global_refit", "coreset", "batch_merge"]
CLASS_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
COLORS      = {"global_refit": "#636EFA", "coreset": "#00CC96",
               "batch_merge": "#EF553B", "full_batch": "#AB63FA"}

rng = np.random.default_rng(RANDOM_STATE)
print("Imports OK")

## 3 — GPU & environment diagnostics

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  {props.total_memory / 1e9:.1f} GB VRAM")
else:
    print("  WARNING: no GPU found — encoding will be slow")

ram_gb = psutil.virtual_memory().total / 1e9
print(f"System RAM : {ram_gb:.1f} GB")
print(f"Torch : {torch.__version__}")

## 4 — Load AG News

In [ ]:
print("Loading AG News train split...")
ds = load_dataset("ag_news", split="train")

documents  = [row["text"] for row in ds]          # already title + description
labels_raw = np.array([row["label"] for row in ds])

print(f"Total documents : {len(documents):,}")
print(f"Classes         : {[CLASS_NAMES[i] for i in sorted(CLASS_NAMES)]}")
print("\nClass distribution:")
print(pd.Series(labels_raw).value_counts().rename(index=CLASS_NAMES).sort_index())
print(f"\nSample doc: {documents[0][:120]}...")

In [ ]:
counts = pd.Series(labels_raw).value_counts().sort_index()
fig = px.bar(
    x=[CLASS_NAMES[i] for i in counts.index],
    y=counts.values,
    color=[CLASS_NAMES[i] for i in counts.index],
    color_discrete_sequence=["#636EFA", "#EF553B", "#00CC96", "#AB63FA"],
    title="AG News — Class Distribution (120K train)",
    labels={"x": "Class", "y": "Documents"},
)
fig.update_layout(showlegend=False)
fig.show()

## 5 — Drift-aware batch construction

Batches simulate **progressive topic emergence**:  
- Batches 1–3: only World + Sports (2 topics visible)  
- Batches 4–6: Business appears (3 topics)  
- Batches 7–12: Sci/Tech appears, then all 4 topics stable  

This creates two natural drift events that should trigger reclusters at batches 4 and 7.

In [ ]:
def make_agnews_drift_batches(documents, labels, rng, n_batches=12, target_batch_size=10_000):
    """
    Returns a list of index arrays. Each doc appears in exactly once.
    AG News has 30K docs per class × 4 = 120K total.

    Draws per batch (must total ≤ 30K per class across all 12 batches):
      Phase 1 (0-2):  World 5000 + Sports 5000                          → cl0 +=15K, cl1 +=15K
      Phase 2 (3-5):  World 3000 + Sports 3000 + Business 4000          → cl0 +=9K,  cl1 +=9K,  cl2 +=12K
      Phase 3 (6-8):  World 2000 + Sports 2000 + Business 2000 + Sci 4000→ cl0 +=6K,  cl1 +=6K,  cl2 +=6K,  cl3 +=12K
      Phase 4 (9-11): Business 4000 + Sci/Tech 6000                     →                       cl2 +=12K, cl3 +=18K

    Totals: cl0=30K  cl1=30K  cl2=30K  cl3=30K  ✓
    """
    idx_by_class = {c: rng.permutation(np.where(labels == c)[0]) for c in range(4)}
    cursors = {c: 0 for c in range(4)}

    def draw(cls, n):
        arr = idx_by_class[cls]
        s = cursors[cls]
        assert s + n <= len(arr), (
            f"Ran out of class-{cls} docs: need {s+n}, have {len(arr)}. "
            "Adjust draw counts."
        )
        cursors[cls] += n
        return arr[s : s + n]

    batch_indices = []

    # Phase 1: World + Sports only
    for _ in range(3):
        idx = np.concatenate([draw(0, 5000), draw(1, 5000)])
        batch_indices.append(rng.permutation(idx))

    # Phase 2: Business enters
    for _ in range(3):
        idx = np.concatenate([draw(0, 3000), draw(1, 3000), draw(2, 4000)])
        batch_indices.append(rng.permutation(idx))

    # Phase 3: Sci/Tech enters; World+Sports declining
    for _ in range(3):
        idx = np.concatenate([draw(0, 2000), draw(1, 2000), draw(2, 2000), draw(3, 4000)])
        batch_indices.append(rng.permutation(idx))

    # Phase 4: World+Sports gone; only Business + Sci/Tech
    for _ in range(3):
        idx = np.concatenate([draw(2, 4000), draw(3, 6000)])
        batch_indices.append(rng.permutation(idx))

    # Sanity check
    used = {c: cursors[c] for c in range(4)}
    assert all(v == 30_000 for v in used.values()), f"Unexpected draw counts: {used}"

    return batch_indices


batch_indices = make_agnews_drift_batches(documents, labels_raw, rng,
                                          n_batches=N_BATCHES,
                                          target_batch_size=BATCH_SIZE)

batch_docs   = [[documents[i] for i in idx] for idx in batch_indices]
batch_labels = [labels_raw[idx] for idx in batch_indices]

# Ordered corpus — cumulative model and full-batch model must see the same doc order
all_indices      = np.concatenate(batch_indices)
all_docs_ordered = [documents[i] for i in all_indices]
labels_ordered   = labels_raw[all_indices]

print(f"Batches         : {len(batch_docs)}")
print(f"Docs per batch  : {[len(b) for b in batch_docs]}")
print(f"Total docs used : {len(all_docs_ordered):,}")

In [ ]:
# Batch composition heatmap — verify the drift staircase
comp = np.zeros((len(batch_labels), 4))
for i, lbls in enumerate(batch_labels):
    for c in range(4):
        comp[i, c] = (lbls == c).mean()

fig = px.imshow(
    comp.T,
    x=[f"Batch {i+1}" for i in range(len(batch_labels))],
    y=[CLASS_NAMES[c] for c in range(4)],
    color_continuous_scale="Blues",
    title="Batch Composition — Topic Drift Simulation (fraction per class)",
    labels={"color": "Fraction"},
    text_auto=".2f",
    aspect="auto",
)
fig.update_layout(height=350)
fig.show()

## 6 — GPU embedding (cached)

Encode all docs once; reuse across strategies. `normalize_embeddings=True` ensures cosine-compatible representations.

In [ ]:
if EMB_CACHE.exists() and DOCS_CACHE.exists():
    print("Loading cached embeddings...")
    all_embeddings = np.load(EMB_CACHE)
    with open(DOCS_CACHE, "rb") as f:
        _cached_docs = pickle.load(f)
    assert len(_cached_docs) == len(all_docs_ordered), \
        "Cache doc count mismatch — delete cache files and rerun"
    print(f"Loaded from cache: {all_embeddings.shape}")
else:
    print(f"Encoding {len(all_docs_ordered):,} docs with '{EMBED_MODEL}' on {device}...")
    st_model = SentenceTransformer(EMBED_MODEL, device=device)
    t0 = time.time()
    all_embeddings = st_model.encode(
        all_docs_ordered,
        batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    print(f"Encoded in {time.time() - t0:.1f}s  |  shape={all_embeddings.shape}")
    np.save(EMB_CACHE, all_embeddings)
    np.save(LABELS_CACHE, labels_ordered)
    with open(DOCS_CACHE, "wb") as f:
        pickle.dump(all_docs_ordered, f)
    del st_model
    torch.cuda.empty_cache()

# Per-batch embedding slices (numpy views, no extra RAM)
batch_embs, start = [], 0
for idx in batch_indices:
    n = len(idx)
    batch_embs.append(all_embeddings[start : start + n])
    start += n

print(f"Embedding matrix : {all_embeddings.shape}  {all_embeddings.nbytes / 1e6:.1f} MB")

## 7 — TriTopic configuration

In [ ]:
base_config = TriTopicConfig(
    use_dim_reduction=False,         # skip UMAP per-recluster — saves 5-15 min each
    use_lexical_view=True,           # TF-IDF co-occurrence graph; strong signal on news
    use_iterative_refinement=False,  # 3× speedup with minimal quality loss at 10k+ docs
    n_consensus_runs=N_CONSENSUS_RUNS,
    min_cluster_size=MIN_CLUSTER_SIZE_ABS,       # absolute floor
    min_cluster_fraction=MIN_CLUSTER_FRACTION,   # 0.5% of corpus at fit time
    low_memory=True,                 # MUST: prevents N×N co-occurrence matrix at 120K docs
    knn_backend="auto",              # uses HNSW above 5k docs
    n_neighbors=15,
    random_state=RANDOM_STATE,
    verbose=False,
)
# What this means in practice:
#   batch 1 (10K docs):    effective mcs = max(5, 0.005*10000)  =  50
#   after 12 batches (120K): effective mcs = max(5, 0.005*120000) = 600
#   full-batch fit (120K): effective mcs = max(5, 0.005*120000) = 600
print("base_config ready")

## 8 — Benchmark loop: all 3 strategies

Each strategy streams the same 12 batches. Per-batch metrics are collected so we can plot novelty over time and recluster events.

In [ ]:
results_per_batch = {s: [] for s in STRATEGIES}
models = {}

for strategy in STRATEGIES:
    print(f"\n{'='*64}")
    print(f"  Strategy: {strategy}")
    print(f"{'='*64}")

    cfg = CumulativeConfig(
        base_config=copy.deepcopy(base_config),
        strategy=strategy,
        recluster_trigger="drift",
        novelty_threshold=NOVELTY_THRESHOLD,
        min_docs_between_recluster=0,
        verbose=False,
    )
    model = CumulativeTriTopic(cfg)

    header = f"{'batch':>5} | {'n_docs':>7} | {'cumul':>7} | {'novelty':>7} | "\
             f"{'reclust':>7} | {'g-topics':>8} | {'wall_s':>6}"
    print(header)
    print("-" * len(header))

    t_strategy = time.time()
    for b_idx, (docs, emb) in enumerate(zip(batch_docs, batch_embs)):
        t0 = time.time()
        result = model.add_batch(docs, embeddings=emb)
        elapsed = time.time() - t0

        results_per_batch[strategy].append({
            "batch"          : b_idx + 1,
            "strategy"       : strategy,
            "n_new_docs"     : result.n_new_docs,
            "n_total_docs"   : result.n_total_docs,
            "novelty"        : result.novelty,
            "reclustered"    : result.reclustered,
            "epoch"          : result.epoch,
            "n_global_topics": model.n_global_topics,
            "wall_s"         : elapsed,
        })

        nv = f"{result.novelty:.3f}" if result.novelty is not None else "  —  "
        rc = "YES ★" if result.reclustered else "no"
        print(f"{b_idx+1:>5} | {len(docs):>7,} | {result.n_total_docs:>7,} | "
              f"{nv:>7} | {rc:>7} | {model.n_global_topics:>8} | {elapsed:>6.1f}")

    t_total = time.time() - t_strategy
    models[strategy] = model
    print(f"\nTotal: {t_total:.1f}s  |  final n_global_topics={model.n_global_topics}")

df_per_batch = pd.concat(
    [pd.DataFrame(v) for v in results_per_batch.values()],
    ignore_index=True,
)
print("\nBenchmark loop complete.")

## 9 — Full-batch TriTopic baseline

Fits a single `TriTopic` on all 120K docs at once — the quality **ceiling**. Every cumulative metric will be interpreted relative to this.

In [ ]:
print(f"Fitting full-batch TriTopic on {len(all_docs_ordered):,} docs...")
t0 = time.time()
full_model = TriTopic(config=copy.deepcopy(base_config))
full_model.fit(all_docs_ordered, embeddings=all_embeddings)
t_full_batch = time.time() - t0

n_full_topics = len([t for t in full_model.topics_ if t.topic_id != -1])
full_ari  = compute_ari(full_model.labels_, labels_ordered)
full_nmi  = compute_nmi(full_model.labels_, labels_ordered)
full_sil  = compute_silhouette(all_embeddings, full_model.labels_)

print(f"\nFull-batch results:")
print(f"  Time        : {t_full_batch:.1f}s")
print(f"  Topics found: {n_full_topics}")
print(f"  ARI vs truth: {full_ari:.4f}")
print(f"  NMI vs truth: {full_nmi:.4f}")
print(f"  Silhouette  : {full_sil:.4f}")

## 10 — Evaluation: cumulative vs full-batch

In [ ]:
comparison_rows = []

for strategy in STRATEGIES:
    model = models[strategy]
    metrics = compare_to_full_batch(model, full_model, labels_true=labels_ordered)

    strategy_df = df_per_batch[df_per_batch.strategy == strategy]
    metrics["strategy"]      = strategy
    metrics["total_wall_s"]  = strategy_df["wall_s"].sum()
    metrics["n_reclusters"]  = int(strategy_df["reclustered"].sum())

    # Coherence & diversity on a 10K sample (full corpus is slow)
    sample_docs = all_docs_ordered[:10_000]
    topics = [t for t in model.model_.topics_ if t.topic_id != -1]
    coherences = [
        compute_coherence(t.keywords[:5], sample_docs, method="npmi")
        for t in topics
    ]
    metrics["mean_coherence"] = float(np.mean(coherences)) if coherences else 0.0
    all_kws = [kw for t in topics for kw in t.keywords]
    metrics["diversity"] = compute_diversity(all_kws, len(topics))

    comparison_rows.append(metrics)
    print(f"{strategy:15s}  ARI_truth={metrics['ari_vs_truth_cumulative']:.4f}  "
          f"ARI_full={metrics['ari_vs_full']:.4f}  sil={metrics['silhouette_cumulative']:.4f}  "
          f"coh={metrics['mean_coherence']:.4f}  div={metrics['diversity']:.4f}  "
          f"time={metrics['total_wall_s']:.1f}s")

In [ ]:
# Full-batch coherence + diversity
sample_docs = all_docs_ordered[:10_000]
full_topics = [t for t in full_model.topics_ if t.topic_id != -1]
full_coherences = [compute_coherence(t.keywords[:5], sample_docs, method="npmi") for t in full_topics]
full_all_kws = [kw for t in full_topics for kw in t.keywords]

full_row = {
    "strategy"               : "full_batch",
    "ari_vs_truth_cumulative": full_ari,
    "nmi_vs_truth_cumulative": full_nmi,
    "ari_vs_full"            : 1.0,
    "nmi_vs_full"            : 1.0,
    "silhouette_cumulative"  : full_sil,
    "silhouette_full"        : full_sil,
    "silhouette_delta"       : 0.0,
    "n_topics_cumulative"    : n_full_topics,
    "n_topics_full"          : n_full_topics,
    "topic_count_drift"      : 0,
    "keyword_overlap"        : 1.0,
    "outlier_ratio_cumulative": float(np.mean(full_model.labels_ == -1)),
    "outlier_ratio_full"     : float(np.mean(full_model.labels_ == -1)),
    "n_epochs"               : 1,
    "n_docs"                 : len(all_docs_ordered),
    "ari_vs_truth_full"      : full_ari,
    "nmi_vs_truth_full"      : full_nmi,
    "total_wall_s"           : t_full_batch,
    "n_reclusters"           : 1,
    "mean_coherence"         : float(np.mean(full_coherences)) if full_coherences else 0.0,
    "diversity"              : compute_diversity(full_all_kws, len(full_topics)),
}
comparison_rows.append(full_row)

print(f"full_batch       ARI_truth={full_ari:.4f}  ARI_full=1.0000  time={t_full_batch:.1f}s")

## 10b — Fair evaluation: similarity-based topic merging

The models may discover more fine-grained topics than the 4 AG News classes (e.g. "Premier League", "NBA", "Olympics" all within Sports). Raw ARI against 4 ground-truth labels underestimates quality when the model is *correctly* over-segmenting.

We fix this by merging each model down to two target granularities and re-scoring:
- **`target=4`** — matches the 4 AG News ground-truth classes
- **`target=n_full_topics`** — matches the full-batch model's natural resolution (apples-to-apples ceiling)

For `TriTopic`: `reduce_topics()` updates `labels_` in-place on a deep copy.  
For `CumulativeTriTopic`: `labels_` goes stale after `reduce_topics()`, so we do a **nearest-centroid reassignment** over all 120K docs using the merged `topic_embeddings_`.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as _cos_sim


def get_merged_labels(model, all_embeddings, target_n):
    """
    Returns a label array for all docs after merging to target_n topics.

    TriTopic: reduce_topics() updates labels_ in-place on a deep copy.
    CumulativeTriTopic: labels_ goes stale after reduce_topics() because the
      cumulative registry is not automatically updated. Instead we do a
      nearest-centroid reassignment — project all docs to the merged topic
      centroids (topic_embeddings_ after reduction) via cosine similarity.
    """
    if isinstance(model, TriTopic):
        m = copy.deepcopy(model)
        n_before = len([t for t in m.topics_ if t.topic_id != -1])
        if target_n < n_before:
            m.reduce_topics(target_n)
        return m.labels_.copy()

    # CumulativeTriTopic — work on the inner TriTopic
    inner = copy.deepcopy(model.model_)
    n_before = len([t for t in inner.topics_ if t.topic_id != -1])
    if target_n < n_before:
        inner.reduce_topics(target_n)

    centroids = inner.topic_embeddings_     # [n_merged, dim]
    if centroids is None or len(centroids) == 0:
        print(f"  WARNING: no centroids after reduce_topics({target_n}), returning original labels")
        return model.labels_.copy()

    # Nearest-centroid reassignment over all accumulated docs
    sims    = _cos_sim(all_embeddings, centroids)   # [N, n_merged]
    merged  = np.argmax(sims, axis=1).astype(np.int32)
    return merged


# ── run for both target granularities ──────────────────────────────────────
all_models = {"full_batch": full_model, **models}
target_labels = {
    "4 (AG News classes)": 4,
    f"{n_full_topics} (full-batch natural)": n_full_topics,
}

merge_rows = []
for tgt_label, tgt_n in target_labels.items():
    print(f"\nMerging to target_n={tgt_n}  [{tgt_label}]")
    for name, mdl in all_models.items():
        n_original = (
            len([t for t in mdl.topics_ if t.topic_id != -1])
            if isinstance(mdl, TriTopic)
            else mdl.n_global_topics
        )
        if tgt_n >= n_original:
            print(f"  {name}: already has {n_original} topics — skipping reduction")
            lbl = mdl.labels_ if isinstance(mdl, TriTopic) else mdl.labels_
        else:
            lbl = get_merged_labels(mdl, all_embeddings, tgt_n)

        ari = compute_ari(lbl, labels_ordered)
        nmi = compute_nmi(lbl, labels_ordered)
        sil = compute_silhouette(all_embeddings, lbl)
        n_resulting = len(set(lbl[lbl != -1]))
        print(f"  {name:15s}  topics={n_original:3d}→{n_resulting}  ARI={ari:.4f}  NMI={nmi:.4f}  sil={sil:.4f}")
        merge_rows.append({
            "strategy"    : name,
            "target_n"    : tgt_n,
            "target_label": tgt_label,
            "n_original"  : n_original,
            "n_resulting" : n_resulting,
            "ari"         : ari,
            "nmi"         : nmi,
            "silhouette"  : sil,
        })

df_merged = pd.DataFrame(merge_rows)

# ── side-by-side comparison table ──────────────────────────────────────────
pre = (
    pd.DataFrame(comparison_rows)[["strategy", "ari_vs_truth_cumulative", "nmi_vs_truth_cumulative"]]
    .set_index("strategy")
    .rename(columns={"ari_vs_truth_cumulative": "pre_ari", "nmi_vs_truth_cumulative": "pre_nmi"})
)

pivot_rows = {}
for _, row in df_merged.iterrows():
    key = f"merged@{row['target_n']}"
    pivot_rows.setdefault(row["strategy"], {})[f"{key}_ari"] = round(row["ari"], 4)
    pivot_rows.setdefault(row["strategy"], {})[f"{key}_nmi"] = round(row["nmi"], 4)
    pivot_rows.setdefault(row["strategy"], {})[f"{key}_sil"] = round(row["silhouette"], 4)

post = pd.DataFrame(pivot_rows).T
post.index.name = "strategy"

fair_eval = pre.join(post).reindex([s for s in ["full_batch"] + STRATEGIES if s in pre.index])

print("\n── Fair evaluation: pre-merge vs post-merge ──")
green_cols = [c for c in fair_eval.columns if "ari" in c or "nmi" in c]
red_cols   = []
styled_fair = fair_eval.style
for col in green_cols:
    styled_fair = styled_fair.background_gradient(subset=[col], cmap="Greens")
display(styled_fair)

# ── bar chart: pre vs post ARI at target=4 ─────────────────────────────────
chart_data = []
for s in ["full_batch"] + STRATEGIES:
    if s not in fair_eval.index:
        continue
    chart_data.append({"strategy": s, "type": "pre-merge",      "ARI": fair_eval.loc[s, "pre_ari"]})
    chart_data.append({"strategy": s, "type": f"merged@4",      "ARI": fair_eval.loc[s, f"merged@4_ari"]})
    if f"merged@{n_full_topics}_ari" in fair_eval.columns:
        chart_data.append({"strategy": s, "type": f"merged@{n_full_topics}", "ARI": fair_eval.loc[s, f"merged@{n_full_topics}_ari"]})

fig = px.bar(
    pd.DataFrame(chart_data),
    x="strategy", y="ARI", color="type", barmode="group",
    title="ARI vs Ground Truth — Pre-merge vs Post-merge",
    labels={"ARI": "ARI vs AG News labels"},
    color_discrete_sequence=["#d3d3d3", "#636EFA", "#00CC96"],
)
fig.update_layout(height=420)
fig.show()

## 10c — LLM-guided topic grouping (Claude)

`generate_report_themes()` is the LLM equivalent of topic merging: Claude reads all fine-grained topic titles + keywords and **decides** which ones belong together into higher-level meta-themes. The grouping is written to `ReportTheme.topic_ids`, which we use to map each doc → its meta-theme → compute ARI.

**Two stages (both happen inside `bigger_picture()`):**
1. `generate_labels()` — Claude Haiku labels each fine-grained topic with a short title
2. `generate_report_themes()` — a *proposer* call groups topics into N meta-themes; a *narrator* call writes a paragraph per theme

**API key**: add `ANTHROPIC_API_KEY` to Kaggle Secrets (Add-ons → Secrets) before running.  
**Cost**: ~N_topics LLM calls per model. Run on `global_refit` only by default; uncomment the loop to run all strategies.

In [ ]:
import os
from tritopic.labeling.llm_labeler import LLMLabeler

# ── API key: read from Kaggle Secrets ──────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    ANTHROPIC_API_KEY = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
except Exception:
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

if not ANTHROPIC_API_KEY:
    print("No ANTHROPIC_API_KEY found.")
    print("Add it via Add-ons → Secrets in Kaggle, then re-run this cell.")
else:
    labeler = LLMLabeler(
        provider="anthropic",
        api_key=ANTHROPIC_API_KEY,
        model="claude-haiku-4-5-20251001",   # cheapest + fast; swap to sonnet for richer labels
        style="short",
        domain_hint="news articles covering world events, sports, business, and technology",
        cache=True,    # caches by prompt hash — re-runs are free
        verbose=True,
    )

    # ── Run on global_refit (highest quality model) ────────────────────────
    # To run on all strategies, replace the line below with:
    #   for llm_strategy in STRATEGIES: llm_model = models[llm_strategy]; ...
    llm_strategy = "global_refit"
    llm_model    = models[llm_strategy]

    print(f"\nRunning bigger_picture() on '{llm_strategy}' "
          f"({llm_model.n_global_topics} topics)...")
    print("Stage 1: generate_labels() — one LLM call per topic")
    print("Stage 2: generate_report_themes() — proposer + narrator calls")

    view = llm_model.bigger_picture(labeler=labeler, n_levels=3, n_themes=4)
    themes = view["themes"]   # list[ReportTheme]

    # ── Theme table ────────────────────────────────────────────────────────
    print(f"\n{'─'*70}")
    print(f"  Meta-themes found: {len(themes)}")
    print(f"{'─'*70}")
    theme_rows = []
    for th in themes:
        print(f"\n  Theme {th.theme_id}: {th.title}")
        print(f"  Docs : {th.total_size:,}  |  Sub-topics: {len(th.topic_ids)}")
        print(f"  Keywords: {', '.join(th.keywords[:8])}")
        print(f"  {th.narrative[:200]}...")
        theme_rows.append({
            "theme_id"  : th.theme_id,
            "title"     : th.title,
            "n_subtopics": len(th.topic_ids),
            "n_docs"    : th.total_size,
            "keywords"  : ", ".join(th.keywords[:6]),
            "subtopic_ids": th.topic_ids,
        })

    df_themes = pd.DataFrame(theme_rows)
    display(df_themes[["theme_id", "title", "n_subtopics", "n_docs", "keywords"]])

    # ── Map docs → theme for ARI ───────────────────────────────────────────
    # ReportTheme.topic_ids are LOCAL ids (inner TriTopic).
    # llm_model.labels_ uses GLOBAL ids.
    # Bridge: _local_id_to_global maps local → global.
    local_to_global = getattr(llm_model, "_local_id_to_global", {})
    global_to_theme = {}
    for th in themes:
        for local_tid in th.topic_ids:
            global_tid = local_to_global.get(local_tid, local_tid)
            global_to_theme[global_tid] = th.theme_id

    theme_labels = np.array(
        [global_to_theme.get(int(l), -1) for l in llm_model.labels_]
    )

    llm_ari = compute_ari(theme_labels, labels_ordered)
    llm_nmi = compute_nmi(theme_labels, labels_ordered)
    llm_sil = compute_silhouette(all_embeddings, theme_labels)
    n_unassigned = int((theme_labels == -1).sum())

    print(f"\n── LLM meta-theme evaluation ({llm_strategy}) ──")
    print(f"  ARI vs ground truth : {llm_ari:.4f}")
    print(f"  NMI vs ground truth : {llm_nmi:.4f}")
    print(f"  Silhouette          : {llm_sil:.4f}")
    print(f"  Docs unassigned (-1): {n_unassigned:,}  "
          f"({100*n_unassigned/len(theme_labels):.1f}% — outlier topics not in any theme)")

    # Compare: pre-merge vs similarity-merge@4 vs LLM-theme
    pre_ari   = float(fair_eval.loc[llm_strategy, "pre_ari"])
    sim4_ari  = float(fair_eval.loc[llm_strategy, "merged@4_ari"])
    comparison_llm = pd.DataFrame([
        {"method": "raw (no merge)",         "ARI": pre_ari},
        {"method": "similarity merge @ 4",   "ARI": sim4_ari},
        {"method": "LLM meta-themes (Claude)","ARI": llm_ari},
    ])
    fig = px.bar(
        comparison_llm, x="method", y="ARI",
        color="method",
        color_discrete_sequence=["#d3d3d3", "#636EFA", "#AB63FA"],
        title=f"Evaluation Methods Compared — {llm_strategy}",
        labels={"ARI": "ARI vs AG News ground truth"},
    )
    fig.add_hline(y=full_ari, line_dash="dot", line_color="black",
                  annotation_text=f"full_batch ceiling ({full_ari:.3f})")
    fig.update_layout(showlegend=False, height=400)
    fig.show()

    # ── Show labelled topics (what Claude called each fine-grained topic) ──
    inner = llm_model.model_
    labelled = [(t.topic_id, getattr(t, "label", "—"), ", ".join(t.keywords[:5]))
                for t in inner.topics_ if t.topic_id != -1]
    df_labels = pd.DataFrame(labelled, columns=["topic_id", "llm_label", "top_keywords"])
    print(f"\nClaude's labels for {len(df_labels)} fine-grained topics:")
    display(df_labels)

## 11 — Visualisations
### 11a — Novelty score over batches

In [ ]:
df_novelty = df_per_batch[df_per_batch.novelty.notna()].copy()

fig = px.line(
    df_novelty,
    x="batch", y="novelty", color="strategy",
    color_discrete_map=COLORS,
    markers=True,
    title="Novelty Score per Batch (fraction of new docs assigned as outliers)",
    labels={"novelty": "Novelty", "batch": "Batch"},
)
# Mark the two drift events
fig.add_vline(x=3.5, line_dash="dash", line_color="gray", annotation_text="Business enters")
fig.add_vline(x=6.5, line_dash="dash", line_color="gray", annotation_text="Sci/Tech enters")
fig.add_hline(y=NOVELTY_THRESHOLD, line_dash="dot", line_color="red",
              annotation_text=f"Recluster threshold ({NOVELTY_THRESHOLD})")
fig.update_layout(height=450)
fig.show()

### 11b — Recluster events timeline

In [ ]:
events = df_per_batch[df_per_batch.reclustered].copy()

fig = px.scatter(
    events,
    x="batch", y="strategy",
    color="strategy",
    color_discrete_map=COLORS,
    symbol_sequence=["star"],
    size_max=20,
    title="Recluster Events by Strategy",
    labels={"batch": "Batch", "strategy": "Strategy"},
)
fig.add_vline(x=3.5, line_dash="dash", line_color="gray")
fig.add_vline(x=6.5, line_dash="dash", line_color="gray")
fig.update_traces(marker_size=18)
fig.update_layout(height=350)
fig.show()

# Counts
print("Recluster counts:")
print(df_per_batch.groupby("strategy")["reclustered"].sum().rename("n_reclusters"))

### 11c — Topic count evolution

In [ ]:
fig = px.line(
    df_per_batch,
    x="batch", y="n_global_topics", color="strategy",
    color_discrete_map=COLORS,
    markers=True,
    title="Global Topic Count Evolution",
    labels={"n_global_topics": "# Global Topics", "batch": "Batch"},
)
fig.add_vline(x=3.5, line_dash="dash", line_color="gray", annotation_text="Business enters")
fig.add_vline(x=6.5, line_dash="dash", line_color="gray", annotation_text="Sci/Tech enters")
# Expected ceiling — 4 AG News classes
fig.add_hline(y=4, line_dash="dot", line_color="black", annotation_text="4 AG News classes")
fig.update_layout(height=400)
fig.show()

### 11d — Wall-clock time comparison

In [ ]:
timing = df_per_batch.groupby("strategy")["wall_s"].agg(
    total_s="sum", mean_s_per_batch="mean"
).reset_index()
# Add full-batch row
timing = pd.concat([
    timing,
    pd.DataFrame([{"strategy": "full_batch", "total_s": t_full_batch,
                   "mean_s_per_batch": t_full_batch}])
], ignore_index=True)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Total wall-clock time (s)", "Avg time per batch (s)"))
for trace_col, row, col in [("total_s", 1, 1), ("mean_s_per_batch", 1, 2)]:
    fig.add_trace(
        go.Bar(
            x=timing["strategy"],
            y=timing[trace_col],
            marker_color=[COLORS.get(s, "#999") for s in timing["strategy"]],
            showlegend=False,
        ),
        row=row, col=col,
    )
fig.update_layout(title="Wall-Clock Time per Strategy", height=400)
fig.show()

### 11e — Side-by-side document map (5K sample, shared UMAP)

Same 5K documents coloured by each model's topic assignments. Visual check that cluster boundaries agree.

In [ ]:
import umap as umap_lib

SAMPLE_N = 5_000
sample_idx = rng.choice(len(all_docs_ordered), size=SAMPLE_N, replace=False)
sample_emb = all_embeddings[sample_idx]

print(f"Running UMAP on {SAMPLE_N} sample docs...")
t0 = time.time()
reducer = umap_lib.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        metric="cosine", random_state=RANDOM_STATE, n_jobs=1)
emb_2d = reducer.fit_transform(sample_emb)
print(f"UMAP done in {time.time() - t0:.1f}s")

all_models_labels = {
    "full_batch"  : full_model.labels_[sample_idx],
    "global_refit": models["global_refit"].labels_[sample_idx],
    "coreset"     : models["coreset"].labels_[sample_idx],
    "batch_merge" : models["batch_merge"].labels_[sample_idx],
}

panel_order = ["full_batch", "global_refit", "coreset", "batch_merge"]
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f"{s} (ARI vs truth: "
                                    f"{compute_ari(all_models_labels[s], labels_ordered[sample_idx]):.3f})"
                                    for s in panel_order])
positions = [(1,1),(1,2),(2,1),(2,2)]

for (row, col), strategy in zip(positions, panel_order):
    lbl = all_models_labels[strategy].astype(str)
    fig.add_trace(
        go.Scatter(
            x=emb_2d[:, 0], y=emb_2d[:, 1],
            mode="markers",
            marker=dict(size=3, color=all_models_labels[strategy], colorscale="Viridis",
                        showscale=False),
            name=strategy,
            showlegend=False,
        ),
        row=row, col=col,
    )

fig.update_layout(title="Document Maps (5K sample) — Shared UMAP Projection",
                  height=750)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

### 11f — Topic keywords per strategy

In [ ]:
for strategy in STRATEGIES + ["full_batch"]:
    m = models[strategy] if strategy != "full_batch" else full_model
    inner = m.model_ if hasattr(m, "model_") else m
    topics_list = [t for t in inner.topics_ if t.topic_id != -1]
    print(f"\n─── {strategy} ({len(topics_list)} topics) ───")
    for t in topics_list:
        print(f"  Topic {t.topic_id:3d}: {', '.join(t.keywords[:8])}")

In [ ]:
def plot_topic_keywords(model_or_topics, title, n_keywords=8, max_topics=30):
    """
    Robust keyword bar chart that works for any number of topics.

    Uses visualize_topics() when topic count is small enough for Plotly subplots
    (≤ max_topics), otherwise falls back to a single grouped horizontal bar chart.
    """
    # Resolve topics list
    if hasattr(model_or_topics, "model_"):               # CumulativeTriTopic
        inner = model_or_topics.model_
    elif hasattr(model_or_topics, "topics_"):            # TriTopic
        inner = model_or_topics
    else:
        inner = model_or_topics
    topics_list = [t for t in inner.topics_ if t.topic_id != -1]
    n_topics = len(topics_list)

    if n_topics == 0:
        print(f"{title}: no non-outlier topics found")
        return

    if n_topics <= max_topics:
        # Use the built-in method — it creates one subplot per topic
        try:
            if hasattr(model_or_topics, "visualize_topics"):
                fig = model_or_topics.visualize_topics(n_keywords=n_keywords)
            else:
                from tritopic.visualization.plotter import TopicVisualizer
                fig = TopicVisualizer().plot_topics(topics=topics_list, n_keywords=n_keywords)
            fig.update_layout(title=title)
            fig.show()
            return
        except ValueError:
            pass  # fall through to custom chart

    # Fallback: single horizontal bar chart, top max_topics topics sorted by size
    print(f"{title}: {n_topics} topics — showing top {max_topics} by keyword count")
    topics_sorted = sorted(topics_list, key=lambda t: -len(t.keywords))[:max_topics]

    rows, kws, scores = [], [], []
    for t in topics_sorted:
        label = f"T{t.topic_id}"
        for rank, kw in enumerate(t.keywords[:n_keywords]):
            rows.append(label)
            kws.append(kw)
            scores.append(n_keywords - rank)   # weight by rank (top kw = highest bar)

    df_kw = pd.DataFrame({"topic": rows, "keyword": kws, "score": scores})
    fig = px.bar(
        df_kw, x="score", y="keyword", color="topic",
        orientation="h",
        title=f"{title} ({n_topics} topics total, showing top {max_topics})",
        labels={"score": "Rank weight", "keyword": "Keyword"},
        height=max(400, min(max_topics * n_keywords * 14, 1200)),
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    fig.show()


for strategy in STRATEGIES:
    plot_topic_keywords(models[strategy], title=f"Topic Keywords — {strategy}",
                        n_keywords=8, max_topics=30)

plot_topic_keywords(full_model, title="Topic Keywords — full_batch (ceiling)",
                    n_keywords=8, max_topics=30)

### 11g — Intertopic distance maps

In [ ]:
for strategy in STRATEGIES:
    try:
        fig = models[strategy].visualize_topic_map(method="mds")
        fig.update_layout(title=f"Intertopic Distance Map — {strategy}")
        fig.show()
    except Exception as e:
        print(f"{strategy}: visualize_topic_map failed — {e}")

### 11h — Topic hierarchy dendrograms

In [ ]:
for strategy in STRATEGIES:
    try:
        fig = models[strategy].visualize_hierarchy()
        fig.update_layout(title=f"Topic Hierarchy — {strategy}")
        fig.show()
    except Exception as e:
        print(f"{strategy}: visualize_hierarchy failed — {e}")

### 11i — Topic overlap heatmap (global_refit)

In [ ]:
try:
    fig = models["global_refit"].visualize_overlap(threshold=0.05)
    fig.update_layout(title="Topic Co-occurrence — global_refit")
    fig.show()
except Exception as e:
    print(f"visualize_overlap failed — {e}")

## 12 — Summary table

In [ ]:
show_cols = [
    "strategy",
    "ari_vs_truth_cumulative",
    "nmi_vs_truth_cumulative",
    "ari_vs_full",
    "nmi_vs_full",
    "silhouette_cumulative",
    "keyword_overlap",
    "n_topics_cumulative",
    "topic_count_drift",
    "outlier_ratio_cumulative",
    "mean_coherence",
    "diversity",
    "n_reclusters",
    "total_wall_s",
]

summary_df = (
    pd.DataFrame(comparison_rows)
    .reindex(columns=show_cols)
    .set_index("strategy")
    .round(4)
)

# Ordering: full_batch at top as reference
order = ["full_batch"] + STRATEGIES
summary_df = summary_df.reindex([s for s in order if s in summary_df.index])

# Styled display
quality_cols = [
    "ari_vs_truth_cumulative", "nmi_vs_truth_cumulative",
    "ari_vs_full", "nmi_vs_full",
    "silhouette_cumulative", "keyword_overlap",
    "mean_coherence", "diversity",
]
cost_cols = ["topic_count_drift", "outlier_ratio_cumulative", "total_wall_s"]

styled = summary_df.style
for col in quality_cols:
    if col in summary_df.columns:
        styled = styled.background_gradient(subset=[col], cmap="Greens")
for col in cost_cols:
    if col in summary_df.columns:
        styled = styled.background_gradient(subset=[col], cmap="Reds_r")

display(styled)

## 13 — Radar chart: quality vs speed

In [ ]:
radar_metrics = [
    "ari_vs_truth_cumulative",
    "nmi_vs_truth_cumulative",
    "silhouette_cumulative",
    "mean_coherence",
    "diversity",
    "speed",  # synthetic: 1 / total_wall_s (normalised below)
]

radar_df = summary_df.copy()
# Normalise silhouette and coherence to [0, 1] against the column range
for col in ["silhouette_cumulative", "mean_coherence", "ari_vs_truth_cumulative",
            "nmi_vs_truth_cumulative", "diversity"]:
    col_min, col_max = radar_df[col].min(), radar_df[col].max()
    if col_max > col_min:
        radar_df[col] = (radar_df[col] - col_min) / (col_max - col_min)
    else:
        radar_df[col] = 1.0

# Speed: inverse of total_wall_s, normalised
inv_time = 1.0 / (radar_df["total_wall_s"] + 1e-6)
radar_df["speed"] = (inv_time - inv_time.min()) / (inv_time.max() - inv_time.min() + 1e-9)

fig = go.Figure()
labels = radar_metrics

for strategy in order:
    if strategy not in radar_df.index:
        continue
    values = [float(radar_df.loc[strategy, m]) for m in radar_metrics]
    values += [values[0]]  # close the polygon
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=labels + [labels[0]],
        fill="toself",
        name=strategy,
        line_color=COLORS.get(strategy, "#999"),
        opacity=0.7,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Strategy Comparison — Quality vs Speed (normalised to [0,1])",
    height=550,
)
fig.show()

## 14 — Full document maps (expensive — run last)

UMAP on all 120K accumulated docs per strategy. Each call takes ~5–10 min on Kaggle.

In [ ]:
# Uncomment to run — each call is ~5-10 min
# for strategy in STRATEGIES:
#     fig = models[strategy].visualize(method="umap", show_outliers=False, interactive=True)
#     fig.update_layout(title=f"Document Map — {strategy} (all {models[strategy].n_global_topics} topics)")
#     fig.show()

## 15 — Conclusions

Fill in the observed values after running the notebook:

---

### Observed results

| Strategy | ARI vs truth | ARI vs full | Time (s) | Reclusters |
|---|---|---|---|---|
| full_batch | ___ | 1.000 | ___ | 1 |
| global_refit | ___ | ___ | ___ | ___ |
| coreset | ___ | ___ | ___ | ___ |
| batch_merge | ___ | ___ | ___ | ___ |

### Interpretation

**global_refit**: Expected to track drift most accurately — novelty spikes should appear at batches 4 and 7 when new topics enter. ARI vs truth expected ≥ 0.6 given AG News's very distinct topics.

**coreset**: Should match `global_refit` closely (within ~0.05 ARI) at ≈40–60% of the cost.

**batch_merge**: Will over-recluster (novelty measured against a single-batch model, not the full accumulator) and show lower keyword overlap — this is a structural property of the strategy, not a bug. Best suited for latency-critical applications where topic quality is secondary.

### Recommendation

- Use `global_refit` with `recluster_trigger="drift"` for corpora up to ~300K docs where RAM permits.  
- Switch to `coreset` above that limit.  
- Use `batch_merge` only when per-batch latency (not overall quality) is the primary constraint.